# Delay Prediction Model Training

Train and evaluate ML model for predicting aircraft turnaround delays.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import matplotlib.pyplot as plt

## Feature Engineering

In [ ]:
def engineer_features(df):
    """Create features for delay prediction."""
    
    # Aircraft size encoding
    aircraft_size_map = {
        'A321': 3, 'B737': 2, 'A330': 4, 'B777': 4
    }
    df['aircraft_size'] = df['aircraft_type'].map(aircraft_size_map)
    
    # Time-based features
    df['is_peak_hour'] = df['hour_of_day'].isin([7, 8, 9, 17, 18, 19]).astype(int)
    
    # Turnaround complexity
    df['turnaround_complexity'] = (
        df['aircraft_size'] * 1.5 + 
        df['passenger_count'] / 50
    )
    
    return df

## Model Training

In [ ]:
# Prepare data
feature_cols = ['aircraft_size', 'hour_of_day', 'passenger_count', 'gate_congestion', 'turnaround_complexity', 'is_peak_hour']
target_col = 'delay_minutes'

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    df[feature_cols], 
    df[target_col], 
    test_size=0.2, 
    random_state=42
)

# Train model
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42
)

model.fit(X_train, y_train)

print('Model trained successfully!')

## Model Evaluation

In [ ]:
# Predictions
y_pred = model.predict(X_test)

# Metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'Mean Absolute Error: {mae:.2f} minutes')
print(f'Root Mean Squared Error: {rmse:.2f} minutes')
print(f'R² Score: {r2:.3f}')

## Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance for Delay Prediction')
plt.tight_layout()
plt.show()

## Save Model

In [ ]:
# Save trained model
joblib.dump(model, '../models/delay_predictor.joblib')
print('Model saved to ../models/delay_predictor.joblib')